# Insurance Premium Prediction — CPE232 Data Models (Kaggle)

## Overview
This notebook is designed for the Kaggle environment to participate in the Insurance Premium Prediction Hackathon.

**Goal:** Predict the continuous variable `Premium Amount` using regression techniques.
**Evaluation Metric:** Mean Absolute Error (MAE).

## Pipeline Steps
1.  **Exploratory Data Analysis (EDA):** Understand distributions and correlations.
2.  **Data Preprocessing:** Handle missing values, outliers, and date formats.
3.  **Feature Engineering:** Create new features from existing data (e.g., Policy Duration, Text Length).
4.  **Modeling (Ensemble):** Use a blend of XGBoost, LightGBM, and CatBoost with K-Fold Cross Validation.
5.  **Submission:** Generate the final submission file.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Load Data
Using the specified Kaggle paths.

In [ ]:
# Kaggle Paths
BASE_PATH = "/kaggle/input/competitions/cpe-232-insurance-premium-prediction"
train_path  = f"{BASE_PATH}/train.csv"
test_path   = f"{BASE_PATH}/test.csv"
sample_path = f"{BASE_PATH}/sample_submission.csv"

print(f"Loading data from: {train_path}")
import os

try:
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    sample_sub = pd.read_csv(sample_path)
    print(f"Train Shape: {train.shape}")
    print(f"Test Shape:  {test.shape}")
except FileNotFoundError:
    print("Files not found in Kaggle path. Checking local fallback for testing...")
    # Optional: Local fallback for development on your machine
    LOCAL_PATH = "cpe-232-insurance-premium-prediction"
    if os.path.exists(LOCAL_PATH):
        train = pd.read_csv(f"{LOCAL_PATH}/train.csv")
        test = pd.read_csv(f"{LOCAL_PATH}/test.csv")
        sample_sub = pd.read_csv(f"{LOCAL_PATH}/sample_submission.csv")
        print("Loaded from local path.")
    else:
        print("Please ensure the dataset is attached to the notebook.")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Target Distribution
plt.figure(figsize=(10, 5))
sns.histplot(train['Premium Amount'], bins=50, kde=True, color='#4c72b0')
plt.title('Distribution of Premium Amount')
plt.show()

# Correlation Matrix (Numerical)
num_cols = train.select_dtypes(include=[np.number]).columns.drop(['id', 'Premium Amount'], errors='ignore')
corr = train[num_cols].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

## 4. Preprocessing & Feature Engineering
- **Date Parsing:** Extract meaningful components from `Policy Start Date`.
- **Text Features:** Extract length from `Customer Feedback`.
- **Handling Missing Values:** Median for numerical, Mode for categorical.

In [ ]:
# Clean column names (remove leading/trailing spaces)
train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

def preprocess_date(df):
    date_col = 'Policy Start Date'
    
    # Check if the column exists
    if date_col not in df.columns:
        print(f"Warning: '{date_col}' not found. Searching for alternatives...")
        found = False
        for col in df.columns:
            if 'start date' in col.lower() or 'policy date' in col.lower():
                print(f"Found alternative date column: '{col}'")
                date_col = col
                found = True
                break
        
        if not found:
            print(f"Error: Could not find date column. Available columns: {list(df.columns)}")
            return df
    
    # Keep original column name for consistency or rename? Let's use the found name but process it.
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    
    # Feature Engineering from Date
    df['Policy_Year'] = df[date_col].dt.year
    df['Policy_Month'] = df[date_col].dt.month
    df['Policy_Day'] = df[date_col].dt.day
    df['Policy_DayOfWeek'] = df[date_col].dt.dayofweek
    
    # Calculate Policy Duration in Days (relative to the latest date in dataset)
    ref_date = df[date_col].max()
    df['Policy_Age_Days'] = (ref_date - df[date_col]).dt.days
    
    return df

def feature_engineering(df):
    if 'Customer Feedback' in df.columns:
        # Text length feature
        df['Feedback_Len'] = df['Customer Feedback'].astype(str).apply(len)
    else:
         print("Warning: 'Customer Feedback' col not found, skipping feature.")
    return df

print("Processing Features...")
train = preprocess_date(train)
test = preprocess_date(test)

train = feature_engineering(train)
test = feature_engineering(test)

# Drop Columns that won't be used for training
# Note: Need to drop the actual date column found above if it wasn't renamed.
# For safety, let's drop any column with 'Date' in it that we processed, or specific list.
drop_cols = ['id', 'Customer Feedback', 'Policy Start Date']
# Ensure columns exist before dropping
cols_to_drop = [c for c in drop_cols if c in train.columns]

X = train.drop(columns=['Premium Amount'] + cols_to_drop, errors='ignore')
y = train['Premium Amount']
X_test = test.drop(columns=cols_to_drop, errors='ignore')

# Imputation
num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numerical Features: {len(num_features)}")
print(f"Categorical Features: {len(cat_features)}")

# Numerical -> Median
if len(num_features) > 0:
    imputer_num = SimpleImputer(strategy='median')
    X[num_features] = imputer_num.fit_transform(X[num_features])
    X_test[num_features] = imputer_num.transform(X_test[num_features])

# Categorical -> Most Frequent
if len(cat_features) > 0:
    imputer_cat = SimpleImputer(strategy='most_frequent')
    X[cat_features] = imputer_cat.fit_transform(X[cat_features])
    X_test[cat_features] = imputer_cat.transform(X_test[cat_features])

# Encoding Categorical Variables
# Label Encoding is effective for Tree-based models (XGB/LGB/CatBoost)
for col in cat_features:
    le = LabelEncoder()
    # Fit on both train and test to cover all categories
    full_data = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(full_data)
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

print("Preprocessing Complete.")

## 5. Model Training (Ensemble Strategy)
We will train 3 powerful gradient boosting models using **5-Fold Cross Validation**.
1. **XGBoost**
2. **LightGBM**
3. **CatBoost**

The final prediction will be a **Weighted Average** of these models.

In [ ]:
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Global placeholders for storage
oof_preds_xgb = np.zeros(len(X))
test_preds_xgb = np.zeros(len(X_test))

oof_preds_lgb = np.zeros(len(X))
test_preds_lgb = np.zeros(len(X_test))

oof_preds_cat = np.zeros(len(X))
test_preds_cat = np.zeros(len(X_test))

# --- 1. XGBoost ---
print("\n========== Training XGBoost (GPU) ==========")
xgb_params = {
    'n_estimators': 2000,
    'learning_rate': 0.05,
    'max_depth': 8,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'reg:absoluteerror',
    'n_jobs': -1,
    'random_state': 42,
    'eval_metric': 'mae',
    'early_stopping_rounds': 100,
    # GPU Configuration for XGBoost 2.0+
    'tree_method': 'hist', 
    'device': 'cuda' 
}

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = xgb.XGBRegressor(**xgb_params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    
    oof_preds_xgb[val_idx] = model.predict(X_val)
    test_preds_xgb += model.predict(X_test) / N_FOLDS
    print(f"Fold {fold+1} MAE: {mean_absolute_error(y_val, oof_preds_xgb[val_idx]):.4f}")

mae_xgb = mean_absolute_error(y, oof_preds_xgb)
print(f"XGBoost Overall MAE: {mae_xgb:.4f}")


# --- 2. LightGBM ---
print("\n========== Training LightGBM ==========")
lgb_params = {
    'n_estimators': 2000,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'objective': 'mae',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = lgb.LGBMRegressor(**lgb_params)
    
    callbacks = [lgb.early_stopping(stopping_rounds=100, verbose=False)]
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric='mae', callbacks=callbacks)
    
    oof_preds_lgb[val_idx] = model.predict(X_val)
    test_preds_lgb += model.predict(X_test) / N_FOLDS
    print(f"Fold {fold+1} MAE: {mean_absolute_error(y_val, oof_preds_lgb[val_idx]):.4f}")

mae_lgb = mean_absolute_error(y, oof_preds_lgb)
print(f"LightGBM Overall MAE: {mae_lgb:.4f}")


# --- 3. CatBoost ---
print("\n========== Training CatBoost (GPU) ==========")
cat_params = {
    'iterations': 2000,
    'learning_rate': 0.05,
    'depth': 8,
    'loss_function': 'MAE',
    'verbose': 0,
    'random_state': 42,
    'task_type': 'GPU', # Enable GPU
    'devices': '0'
}

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = CatBoostRegressor(**cat_params)
    model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=100)
    
    oof_preds_cat[val_idx] = model.predict(X_val)
    test_preds_cat += model.predict(X_test) / N_FOLDS
    print(f"Fold {fold+1} MAE: {mean_absolute_error(y_val, oof_preds_cat[val_idx]):.4f}")

mae_cat = mean_absolute_error(y, oof_preds_cat)
print(f"CatBoost Overall MAE: {mae_cat:.4f}")

## 6. Blending & Submission
We assign weights based on inverse MAE performance (better models contribute more).

In [ ]:
# Calculate weights (Inverse variance weighing logic, simplified)
w_xgb = (1/mae_xgb) / ((1/mae_xgb) + (1/mae_lgb) + (1/mae_cat))
w_lgb = (1/mae_lgb) / ((1/mae_xgb) + (1/mae_lgb) + (1/mae_cat))
w_cat = (1/mae_cat) / ((1/mae_xgb) + (1/mae_lgb) + (1/mae_cat))

print(f"Weights -> XGB: {w_xgb:.3f}, LGB: {w_lgb:.3f}, Cat: {w_cat:.3f}")

final_oof_preds = (w_xgb * oof_preds_xgb) + (w_lgb * oof_preds_lgb) + (w_cat * oof_preds_cat)
ensemble_mae = mean_absolute_error(y, final_oof_preds)
print(f"Ensemble OOF MAE: {ensemble_mae:.4f}")

# Final Test Predictions
final_preds = (w_xgb * test_preds_xgb) + (w_lgb * test_preds_lgb) + (w_cat * test_preds_cat)

# Create Submission File
submission = pd.DataFrame({
    'id': sample_sub['id'],
    'Premium Amount': final_preds
})

submission.to_csv('submission.csv', index=False)
print("Submission saved successfully: submission.csv")
submission.head()